# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio





Cloning into 'kauldron'...
remote: Enumerating objects: 9621, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 9621 (delta 56), reused 48 (delta 34), pack-reused 9436 (from 3)
Receiving objects: 100% (9621/9621), 2.88 MiB | 32.37 MiB/s, done.
Resolving deltas: 100% (6906/6906), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 59.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 k

In [2]:
# !pip install -q kauldron
# # Clone the gemma repository if not already present
# !pip install -q gemma

# # Clone the dialog repository to fix Gemma format issues
# !pip install -q dialog

# !pip install -q flax optax seqio

In [3]:
# %rm -rf /content/Titans_jax

In [4]:
!git clone https://github.com/andrew-veriga/Titans_jax.git


Cloning into 'Titans_jax'...
remote: Enumerating objects: 500, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 500 (delta 21), reused 25 (delta 14), pack-reused 463 (from 1)
Receiving objects: 100% (500/500), 182.18 MiB | 37.78 MiB/s, done.
Resolving deltas: 100% (290/290), done.


In [5]:
!pip install importlib_resources

# Start

In [5]:
# !git pull

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
import os
# os._exit(0)

In [1]:
# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp
import dataclasses

# Gemma imports
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


/content/Titans_jax
JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

In [3]:
!rm saved_titans_delta.zip

In [4]:
import shutil
import os

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
titans_zip = "saved_titans_delta.zip"

if os.path.exists(titans_zip):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

if os.path.exists(titans_delta_path):
    # Load original Gemma 3 1B IT weights
    print("Loading original Gemma weights...")
    original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

    # Merge loaded Titans weights into original Gemma params
    print("Merging Titans weights...")
    import titans_tree_utils
    loaded_titans_params = load_titans_weights(titans_delta_path)
    merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
    print("✅ Success: Pre-trained Titans weights loaded and merged.")
else:
    print("⚠️ Note: './saved_titans_delta' not found. Titans layers will be randomly initialized.")

Loading original Gemma weights...


Merging Titans weights...
✅ Success: Pre-trained Titans weights loaded and merged.


## 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Instead of a python generator, we use `kd.data.py.Elements`.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

### hyper-parameters

In [5]:
# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 16
max_length = 1536
total_steps = 40000


In [6]:
# Define the model configuration
gemma_config = Gemma3_1B_Titans.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config, # По умолчанию is_training_mode=True
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x
def format_triviaqa(x):
    # В trivia_qa/rc структура: 'entity_pages' (list) или 'search_results'
    # Берем первый попавшийся контекст из entity_pages
    ctx = ""
    if x['entity_pages']['wiki_context']:
        ctx = x['entity_pages']['wiki_context'][0]

    q = x["question"]
    # Ответ в TriviaQA обычно в x['answer']['value']
    ans = x['answer']['value']

    x['src'] = f"Context: {ctx}\n\nQuestion: {q}"
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


In [7]:
# 1. Выбираем TyDi QA
def get_train_dataset_tydi_qa():
    return kd.data.py.Tfds(
        name= "trivia_qa/rc", #splits: 'test'	17,210 | 'train'	138,384 | 'validation'	18,669
        split="train",
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        num_workers=1,
        transforms=[
            # поля context/question/answers
            format_triviaqa, # format_squad,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target", "loss_mask"]),
        ],
    )

# 2. Настраиваем расписание с Warmup

lr_sched = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=5e-5, # Снижаем пик
    warmup_steps=2000, # Даем время на адаптацию Titans
    decay_steps=total_steps - 2000,
    end_value=1e-6
)

# 3. Оптимизатор с Gradient Clipping и AdamW
optimizer_adamw = optax.MultiSteps(
    kd.optim.partial_updates(
        optax.chain(
            optax.clip_by_global_norm(5.0),# Спасает от шума на графике
            optax.adamw(learning_rate=1e-6, weight_decay=1e-4),
        ),
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=8,
)

In [8]:
import os

cpu_count = os.cpu_count()
print(f"Количество доступных ядер CPU: {cpu_count}")
print(f"Рекомендуемое значение num_workers: от 1 до {max(1, cpu_count // 2)}")

Количество доступных ядер CPU: 44
Рекомендуемое значение num_workers: от 1 до 22


In [9]:
# merged_params = None

In [10]:
# --- INITIALIZATION LOGIC ---
# We define a simple class because Kauldron expects an object with a .transform() method
import flax
from kauldron import metrics as kd_metrics

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        # Состояние может быть пустым, так как данные мы берем напрямую из JAX на хосте
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    # Если словарь пустой, пропускаем устройство
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    # 1. Текущее использование памяти (если ключа нет, вернет 0)
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    # 2. Пиковое использование
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    # 3. Лимит и процент (только если 'limit' действительно существует)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            """Создает пустое начальное состояние."""
            return cls()
        def merge(self, other):
            """Объединяет состояния из разных батчей (здесь ничего не делаем)."""
            return self

    def get_state(self, **kwargs) -> State:
        # Просто возвращаем пустое состояние.
        # Нам не нужны данные из batch или модели для этой метрики.
        return self.State().empty()

# Генерируем словарь для Kauldron

Titan_losses = {
# Kauldron понимает синтаксис [ключ] для доступа к словарям внутри PyTree
    f"distill_layer_{i}": kd.losses.Value(
        values=f"preds.layer_losses['loss_layer_{i}']"
    )
    for i in Gemma3_1B_Titans.config.titans_layer_indices
}

# 2. Метрики для графиков (собирают сырой MSE)
Titan_metrics = {
    # f"raw_mse_{i}": kd.metrics.SingleDimension(
    #     tensor=f"preds.layer_losses['raw_mse_layer_{i}']", # Тут аргумент называется tensor, а не values
    #     index=None # Отключаем срез по индексу, чтобы класс взял всё число целиком
    # )
    # for i in Gemma3_1B_Titans.config.titans_layer_indices
}
Titan_metrics["tpu_memory"]= TPUMemoryMetric()
import flax.linen as nn


# Configure Trainer
trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath('./titans_workdir_squad'),
    train_ds=get_train_dataset_tydi_qa(), ##get_train_dataset(),
    model= model,
    # If we have merged_params from Cell 1, use them directly (much faster).
    # Otherwise, use SkipTitans to load Gemma and randomly init Titans.
    init_transform = FullParamsInit(merged_params) if merged_params is not None else SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        ),
        workdir=os.path.abspath('./SkipTitans_workdir')
    ),
    num_train_steps = total_steps, #87600 // 16, #dataset length times to epoches num divide to batch_size
    train_losses=Titan_losses,
    train_metrics=Titan_metrics,
    optimizer = optimizer_adamw,

    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),

    # Sharding for TPU / TPU Pods
    sharding=kd.sharding.ShardingStrategy(),
    # train_metrics={

    #     # Наш монитор памяти
    #     "tpu_memory": TPUMemoryMetric()
    # },
)

print("Trainer initialized. Starting compilation and training...")


Trainer initialized. Starting compilation and training...


In [11]:
Titan_metrics

{'tpu_memory': <__main__.TPUMemoryMetric at 0x7e70f62a7ef0>}

### 2.5 Monitoring with TensorBoard
Launch TensorBoard to monitor training metrics (Loss, Learning Rate) in real-time.

In [12]:
%reload_ext tensorboard


In [13]:
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()

Known TensorBoard instances:
  - port 6007: logdir ./titans_workdir_squad/ (started 1:58:21 ago; pid 44873)


In [14]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./titans_workdir_squad/ --port=6007

## Start training

In [16]:
# run the actual training loop:
state, aux = trainer.train()

Disabling pygrain multi-processing (unsupported in colab).
Starting training loop at step 20000


train:  50%|####9     | 20000/40001 [00:00<?, ?it/s]

In [ ]:
import os
# os._exit(0)

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [24]:
# --- ЭКСТРЕННОЕ СОХРАНЕНИЕ В GITHUB ---
# Используйте эту ячейку, если обучение прервалось или зависло,
# и вам нужно забрать последние чекпоинты и логи для продолжения.

import os
import shutil
from google.colab import userdata

def emergency_save_and_push(workdir='./titans_workdir_squad'):
    # 1. Проверка наличия данных
    if not os.path.exists(workdir):
        print(f"❌ Директория {workdir} не найдена.")
        return

    zip_name = "titans_emergency_backup.zip"
    if os.path.exists(zip_name): os.remove(zip_name)

    # 2. Архивируем всё: чекпоинты Orbax + логи TensorBoard
    print(f"📦 Архивирование {workdir}...")
    shutil.make_archive("titans_emergency_backup", 'zip', workdir)

    # 3. Пуш в GitHub
    try:
        # Получаем PAT и URL репозитория
        GITHUB_PAT = userdata.get('GITHUB_PAT')
        repo_url_raw = !git config --get remote.origin.url
        repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")
        auth_url = f"https://{GITHUB_PAT}@{repo_url}.git"

        !git config --global user.email "colab-emergency@google.com"
        !git config --global user.name "Colab Emergency Bot"

        !git add {zip_name}
        !git commit -m "Emergency backup: Step recovery [automated]" || echo "No changes"
        !git push {auth_url} HEAD:main --force
        print(f"🚀 Бэкап {zip_name} успешно отправлен в GitHub.")
    except Exception as e:
        print(f"❌ Ошибка GitHub: {e}. Проверьте секрет GITHUB_PAT.")

# Запуск
emergency_save_and_push()

📦 Архивирование ./titans_workdir_squad...
[main 956beea] Emergency backup: Step recovery [automated]
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 titans_emergency_backup.zip
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 44 threads
Compressing objects: 100% (2/2), done.
error: RPC failed; HTTP 500 curl 22 The requested URL returned error: 500
send-pack: unexpected disconnect while reading sideband packet
Writing objects: 100% (3/3), 4.84 GiB | 84.52 MiB/s, done.
Total 3 (delta 1), reused 1 (delta 0), pack-reused 0
fatal: the remote end hung up unexpectedly
Everything up-to-date
🚀 Бэкап titans_emergency_backup.zip успешно отправлен в GitHub.


In [ ]:
# !cd /content/Titans_jax/titans_workdir_squad/checkpoints && zip -r /content/Titans_jax/ckpt_24000.zip ckpt_24000

# Проверим, что файл создался и посмотрим его размер
# !ls -lh /content/Titans_jax/ckpt_24000.zip

In [17]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # 1. Extract only the Titans weights
    _, titans_params = titans_tree_utils.split_titans_params(state.params)

    import shutil
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    checkpointer = ocp.StandardCheckpointer()

    # 2. Correct Orbax syntax: save(path, state)
    # Passing titans_params as the positional state saves only them.
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:
save_titans_weights(state, "./saved_titans_delta")
loaded_titans_params = load_titans_weights("./saved_titans_delta")
original_params, _ = titans_tree_utils.split_titans_params(state.params)
state = state.replace(params=titans_tree_utils.merge_titans_params(original_params, loaded_titans_params))

def save_training_checkpoint(state: kd.train.TrainState, workdir: str):
    """
    Полное сохранение состояния обучения (веса + оптимизатор + шаг).
    Позволяет возобновить тренировку с того же места.
    """
    import orbax.checkpoint as ocp
    checkpoint_path = os.path.join(workdir, 'checkpoints')

    # Kauldron уже имеет встроенный чекпоинтер в Trainer,
    # но этот метод показывает, как сделать это вручную через Orbax.
    options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_path), ocp.StandardCheckpointer(), options)

    # Сохраняем весь объект state (включая opt_state и step)
    mngr.save(state.step, state)
    mngr.wait_until_finished()
    print(f"✅ Full training checkpoint saved at step {state.step} to {checkpoint_path}")

def load_training_checkpoint(state_template: kd.train.TrainState, workdir: str) -> kd.train.TrainState:
    """
    Восстановление полной сессии обучения.
    """
    import orbax.checkpoint as ocp
    checkpoint_path = os.path.join(workdir, 'checkpoints')

    if not os.path.exists(checkpoint_path):
        print("⚠️ No checkpoint found. Starting from scratch.")
        return state_template

    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_path), ocp.StandardCheckpointer())
    latest_step = mngr.latest_step()

    if latest_step is None:
        return state_template

    # Восстанавливаем состояние, используя state_template как структуру
    restored_state = mngr.restore(latest_step, items=state_template)
    print(f"🚀 Restored training session from step {latest_step}")
    return restored_state

Saved ONLY Titans weights to /content/Titans_jax/saved_titans_delta


In [ ]:
!ls ./saved_titans_delta


In [18]:
import os
import zipfile
import shutil
from google.colab import files

checkpoint_dir = "./saved_titans_delta"

# First, let's check if the directory exists
if not os.path.exists(checkpoint_dir):
    print(f"Error: {checkpoint_dir} does not exist!")
else:
    # Try Colab download first
    try:

        # Create a zip file of the checkpoint
        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Download the zip file
        files.download(zip_filename)
        print(f"✓ Downloaded {zip_filename} via Colab")

    except ImportError:
        # Not in Colab, create zip and provide manual download instructions
        print("Not running in Google Colab. Creating zip file...")

        zip_filename = "saved_titans_delta.zip"

        # Remove old zip if exists
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        # Create zip file
        shutil.make_archive("saved_titans_delta", 'zip', checkpoint_dir)

        # Get file size
        file_size = os.path.getsize(zip_filename) / (1024 * 1024)  # Size in MB

        print(f"\n✓ Created {zip_filename} ({file_size:.2f} MB)")
        print(f"📁 Full path: {os.path.abspath(zip_filename)}")
        print("\nTo download, use one of these methods:")
        print("1. Right-click the file in Jupyter file browser and select 'Download'")
        print("2. Use scp/rsync from your terminal:")
        print(f"   scp user@server:{os.path.abspath(zip_filename)} ./")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded saved_titans_delta.zip via Colab


In [20]:
# --- Push to GitHub using Personal Access Token (PAT) ---
from google.colab import userdata
import os

print("Pushing weights to GitHub...")

zip_filename = "saved_titans_delta.zip"
if os.path.exists(zip_filename):
    try:
        # 1. Get PAT from Colab Secrets (Key icon on the left)
        # Make sure you created a secret named 'GITHUB_PAT'
        GITHUB_PAT = userdata.get('GITHUB_PAT')

        # 2. Get repository info
        # Extract owner/repo from current git config
        repo_url_raw = !git config --get remote.origin.url
        repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")

        # 3. Configure git and push
        !git config --global user.email "colab-automated@google.com"
        !git config --global user.name "Colab Bot"

        # Use PAT in URL for authentication
        authenticated_url = f"https://{GITHUB_PAT}@{repo_url}.git"

        !git add {zip_filename}
        !git commit -m "Update trained Titans weights [automated push]" || echo "No changes to commit"
        !git push {authenticated_url} HEAD:main --force

        print("✅ Success: Weights pushed to GitHub.")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nHint: Create a 'GITHUB_PAT' in Colab Secrets (Key icon) with 'repo' permissions.")
else:
    print(f"❌ Error: {zip_filename} not found. Save weights first.")

Pushing weights to GitHub...
On branch main
Your branch is ahead of 'origin/main' by 3 commits.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	saved_titans_delta/
	titans_workdir_squad/

nothing added to commit but untracked files present (use "git add" to track)
No changes to commit
Enumerating objects: 509, done.
Counting objects: 100% (509/509), done.
Delta compression using up to 44 threads
Compressing objects: 100% (217/217), done.
error: RPC failed; HTTP 500 curl 22 The requested URL returned error: 500
send-pack: unexpected disconnect while reading sideband packet
Writing objects: 100% (509/509), 5.07 GiB | 72.28 MiB/s, done.
Total 509 (delta 292), reused 501 (delta 290), pack-reused 0
fatal: the remote end hung up unexpectedly
Everything up-to-date
✅ Success: Weights pushed to GitHub.


In [24]:
!git add {zip_filename}
!git commit -m "Update trained Titans weights [automated push]" || echo "No changes to commit"
!git push {authenticated_url} HEAD:main --force

On branch main
Your branch is ahead of 'origin/main' by 3 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

no changes added to commit (use "git add" and/or "git commit -a")
No changes to commit
Enumerating objects: 509, done.
Counting objects: 100% (509/509), done.
Delta compression using up to 44 threads
Compressing objects: 100% (217/217), done.
error: RPC failed; HTTP 500 curl 22 The requested URL returned error: 500
send-pack: unexpected disconnect while reading sideband packet
Writing objects: 100% (509/509), 5.07 GiB | 74.63 MiB/s, done.
Total 509 (delta 292), reused 501 (delta 290), pack-reused 0
fatal: the remote end hung up unexpectedly
Everything up-to-date


## КОМПАКТНОЕ ЭКСТРЕННОЕ СОХРАНЕНИЕ В GITHUB

In [22]:
# --- КОМПАКТНОЕ ЭКСТРЕННОЕ СОХРАНЕНИЕ В GITHUB ---
# Сохраняет только ПОСЛЕДНИЙ чекпоинт и логи TensorBoard.
# Размер архива составит ~100 МБ вместо 4 ГБ.

import os
import shutil
import orbax.checkpoint as ocp
from google.colab import userdata

def emergency_save_and_push(workdir='./titans_workdir_squad'):
    checkpoint_root = os.path.join(workdir, 'checkpoints')
    if not os.path.exists(checkpoint_root):
        print(f"❌ Директория {checkpoint_root} не найдена.")
        return

    # 1. Находим последний шаг через Orbax
    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_root), ocp.StandardCheckpointer())
    latest_step = mngr.latest_step()
    if latest_step is None:
        print("❌ Актуальные чекпоинты не найдены.")
        return

    print(f"🔍 Найден последний шаг: {latest_step}")
    
    # 2. Создаем временную папку для чистого экспорта
    export_dir = "./titans_export_temp"
    if os.path.exists(export_dir): shutil.rmtree(export_dir)
    os.makedirs(export_dir)

    # 3. Копируем только ПОСЛЕДНИЙ шаг и метаданные Orbax
    os.makedirs(os.path.join(export_dir, "checkpoints"))
    step_src = os.path.join(checkpoint_root, str(latest_step))
    step_dst = os.path.join(export_dir, "checkpoints", str(latest_step))
    shutil.copytree(step_src, step_dst)
    
    # Копируем служебные файлы Orbax, необходимые для загрузки
    for meta in ['_checkpoint_manager_state.json', 'metadata']:
        meta_src = os.path.join(checkpoint_root, meta)
        if os.path.exists(meta_src):
            dest = os.path.join(export_dir, "checkpoints", meta)
            if os.path.isdir(meta_src): shutil.copytree(meta_src, dest)
            else: shutil.copy(meta_src, dest)

    # 4. Копируем логи TensorBoard (tfevents)
    for item in os.listdir(workdir):
        if item.startswith('events.out.tfevents'):
            shutil.copy(os.path.join(workdir, item), export_dir)

    # 5. Архивируем только экспортную папку
    zip_name = "titans_compact_backup.zip"
    if os.path.exists(zip_name): os.remove(zip_name)
    
    print(f"📦 Создание компактного архива...")
    shutil.make_archive("titans_compact_backup", 'zip', export_dir)
    shutil.rmtree(export_dir)
    
    # 6. Пуш в GitHub
    try:
        GITHUB_PAT = userdata.get('GITHUB_PAT')
        repo_url_raw = !git config --get remote.origin.url
        repo_url = repo_url_raw[0].replace("https://", "").replace("git@github.com:", "").replace(".git", "")
        auth_url = f"https://{GITHUB_PAT}@{repo_url}.git"

        !git config --global user.email "colab-emergency@google.com"
        !git config --global user.name "Colab Emergency Bot"
        !git add {zip_name}
        !git commit -m "Compact backup: Step {latest_step} [automated]" || echo "No changes"
        !git push {auth_url} HEAD:main --force
        print(f"🚀 Бэкап {zip_name} (~100MB) успешно отправлен в GitHub.")
    except Exception as e:
        print(f"❌ Ошибка GitHub: {e}")

# Запуск
emergency_save_and_push()


associative_scan.py		     requirements.txt
gemma_titans.py			     sampling_of_Kauldron_Titans_fixed.ipynb
GemmaTitans_Surgery_Plan.md	     saved_titans_delta
Kauldron_Migration_Plan.md	     saved_titans_delta.zip
Kauldron_SkipTitans_Research.md      titans_ckpts.py
Kauldron_Titans_TPU_v5e_fixed.ipynb  titans_emergency_backup.zip
LICENSE				     titans.py
model_loader.py			     titans_tree_utils.py
__pycache__			     titans_workdir_squad
README.md			     weights_surgery.py
